# EUV Stochastic Printing
Ryan Sapp · CS 427 Project 3 · November 2026

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, KFold, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

## Load data

In [ ]:
rows_full = pd.read_csv("../data/full/rows.csv")
rows_test = pd.read_csv("../data/test/rows.csv")

print("full:", rows_full.shape)
print("test:", rows_test.shape)

## Inspect Columns

In [ ]:
for c in rows_full.columns:
    print(c)

In [ ]:
print(rows_full[["na", "k_amp", "sigma_q_nm"]].nunique())

## Select features and targets

A column qualifies as a model input only if it is something that could be
set before running the process, and is not derivable from the result.
14 of the 40 columns pass, the other 26 fail one of those two tests.

| Excluded | Why |
|---|---|
| `resist_id`, `dose_index`, `seed` | Identifiers, not physical settings. `seed` is the random number generator seed |
| `cd_target_nm`, `mask_cd_nm`, `dose_mjcm2` | Already implied by features I kept. `dose_mjcm2` is `d0_mjcm2 × dose_factor`. |
| `na`, `k_amp`, `sigma_q_nm` | Identical in all 20,160 rows. A column that never varies cannot explain variation. |
| `absorbed_frac`, `photons_abs_per_nm2`, `image_peak`, `nils` | Intermediate physics the simulator computes from the settings. Including them would mean predicting the outcome from settings plus part of the answer. |
| `n_holes`, `n_ok`, `n_missing`, `n_merged`, `n_merge_events`, `n_neighbor_pairs`, `p_merged` | Simulation outputs. Using one output to predict another is leakage. |
| `mean_cd_se`, `sigma_cd_se`, `lcdu_3s_nm` | Derived from the targets themselves — `lcdu_3s_nm` is `3 × sigma_cd_nm`. |

The 14 that remain are exactly the quantities an engineer chooses before
exposing a wafer, which is what makes the trained model usable as a
substitute for running the simulator.

In [16]:
FEATURES = [
    "pitch_nm", "cd_ratio", "mask_bias_nm", "d0_mjcm2",
    "sigma_e_nm", "sigma_a_nm", "qe", "rho_pag", "rho_pg",
    "thickness_nm", "absorptivity_um", "dev_threshold",
    "rho_q",
    "dose_factor",
]

TARGETS = ["p_missing", "mean_cd_nm", "sigma_cd_nm"]

## Build film keys and check for collisions

In [17]:
rows_full["run"] = "full"
rows_test["run"] = "test"

for df in (rows_full, rows_test):
    df["resist_key"] = df["run"] + "_" + df["resist_id"].astype(str)

overlap = set(rows_full["resist_key"]) & set(rows_test["resist_key"])
print("full films:", rows_full["resist_key"].nunique())
print("test films:", rows_test["resist_key"].nunique())
print("overlapping keys:", len(overlap))

full films: 2520
test films: 337
overlapping keys: 0


## Separate training and test data sets

In [18]:
train = rows_full
test  = rows_test

print("train:", train.shape, "| test:", test.shape)

train: (20160, 42) | test: (2696, 42)


## Define targets and row filters

In [ ]:
def masks_for(df):
    return {
        "p_missing":   pd.Series(True, index=df.index),
        "mean_cd_nm":  df["n_ok"] > 0,
        "sigma_cd_nm": df["n_ok"] >= 100,
    }

MASKS = masks_for(train)

for t, m in MASKS.items():
    print(f"rows for {t:14s} {m.sum()}")

print("\nrows where nothing printed:", (train["n_ok"] == 0).sum())
print("mean_cd_nm missing:", train["mean_cd_nm"].isna().sum())